# Day 2 Lab — Preference Tuning a Small Pretrained LM

**Goal:** use a real small pretrained LM and the existing chosen/rejected examples to run a minimal DPO-style update on CPU.

The lab computes reference log-probs once, tunes a small trainable subset, and compares chosen-vs-rejected preference margin before and after.

In [2]:
# If needed, uncomment the install line below in a fresh notebook environment.
# %pip install -q "transformers>=4.41" "accelerate>=0.30" torch

import copy, math, random, re, json
from pprint import pprint
import torch
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(7)
torch.manual_seed(7)
DEVICE = "cpu"
# Keep the default very small for CPU. Swap to Qwen/Qwen2.5-0.5B-Instruct if your laptop has enough RAM.
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
FALLBACK_MODEL = "sshleifer/tiny-gpt2"

def load_small_lm(model_name=MODEL_NAME):
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)
        loaded = model_name
    except Exception as exc:
        print(f"Could not load {model_name}: {exc}\nFalling back to {FALLBACK_MODEL}.")
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
        model = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL)
        loaded = FALLBACK_MODEL
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.to(DEVICE)
    model.train()
    print("Loaded:", loaded)
    return tokenizer, model

def make_tiny_trainable(model, keywords=("lm_head", "embed_tokens", "wte")):
    """Freeze most parameters so CPU training remains quick."""
    for p in model.parameters():
        p.requires_grad = False
    trainable = []
    for name, p in model.named_parameters():
        if any(k in name for k in keywords):
            p.requires_grad = True
            trainable.append(name)
    if not trainable:
        # Generic fallback: unfreeze the last few tensors.
        for name, p in list(model.named_parameters())[-8:]:
            p.requires_grad = True
            trainable.append(name)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable tensors: {len(trainable)} | trainable params: {n_params:,}")
    return trainable

def generate_text(model, tokenizer, prompt, max_new_tokens=48):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def sft_loss(model, tokenizer, prompt, target, max_length=256):
    full = prompt + target
    enc = tokenizer(full, return_tensors="pt", truncation=True, max_length=max_length).to(DEVICE)
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids
    labels = enc.input_ids.clone()
    # Train only completion tokens; mask prompt tokens.
    prompt_len = min(prompt_ids.shape[1], labels.shape[1])
    labels[:, :prompt_len] = -100
    outputs = model(**enc, labels=labels)
    return outputs.loss

def run_sft_steps(model, tokenizer, train_rows, steps=8, lr=5e-4):
    make_tiny_trainable(model)
    opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = []
    for step in range(steps):
        row = train_rows[step % len(train_rows)]
        loss = sft_loss(model, tokenizer, row["prompt"], row["target"])
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(float(loss.detach().cpu()))
    print("losses:", [round(x, 3) for x in losses])
    return losses

## 1. Existing preference examples

In [ ]:
pairs = [
    {"prompt":"Summarize this meeting for the team.",
     "chosen":"Decisions: ship canary. Actions: Alex updates eval, Priya monitors latency. Risk: grounding regressions.",
     "rejected":"Great meeting. Everyone discussed next steps and we should follow up soon."},
    {"prompt":"Route this support case.",
     "chosen":"Route to Billing because the refund SLA is missed and owner is Billing.",
     "rejected":"Ask the customer to wait a few more days."},
    {"prompt":"Summarize refund status for a support manager.",
     "chosen":"Refund approved Jun 5; expected Jun 10. Owner: Billing.",
     "rejected":"The refund is probably being processed. Please check later."},
]

def pref_prompt(row):
    return "Instruction: " + row["prompt"] + "\nAnswer: "

{'chosen': 'Decisions: ship canary. Actions: Alex updates eval, Priya monitors '
           'latency. Risk: grounding regressions.',
 'prompt': 'Summarize this meeting for the team.',
 'rejected': 'Great meeting. Everyone discussed next steps and we should '
             'follow up soon.'}

PROMPT: Summarize this meeting for the team.
Instruction: Summarize this meeting for the team.
Answer: 1. I want to make sure that everyone is on the same page regarding the upcoming meeting. I want to make sure that everyone is on the same page regarding the upcoming meeting so that everyone is on

PROMPT: Route this support case.
Instruction: Route this support case.
Answer: 100

Question: The user is looking for an answer to the following question: What is the best way to prepare for a job interview in the tech industry?
A:

PROMPT: Summarize refund status for a support manager.
Instruction: Summarize refund status for a support manager.
Answer: 1. The answer is:


## 2. Load model and define sequence log-probability

In [5]:
tokenizer, model = load_small_lm()

def completion_logprob(model, tokenizer, prompt, completion, max_length=256):
    full = prompt + completion
    enc = tokenizer(full, return_tensors="pt", truncation=True, max_length=max_length).to(DEVICE)
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids
    labels = enc.input_ids.clone()
    labels[:, :min(prompt_ids.shape[1], labels.shape[1])] = -100
    with torch.set_grad_enabled(model.training):
        out = model(**enc)
        logp = torch.log_softmax(out.logits[:, :-1, :], dim=-1)
        target = labels[:, 1:]
        mask = target.ne(-100)
        safe_target = target.masked_fill(~mask, 0)
        token_logp = logp.gather(-1, safe_target.unsqueeze(-1)).squeeze(-1)
        return (token_logp * mask).sum() / mask.sum().clamp(min=1)

def preference_report(model, title):
    model.eval()
    rows = []
    with torch.no_grad():
        for p in pairs:
            pr = pref_prompt(p)
            c = completion_logprob(model, tokenizer, pr, p["chosen"])
            r = completion_logprob(model, tokenizer, pr, p["rejected"])
            rows.append(float((c-r).cpu()))
    print(title, "margins:", [round(x, 3) for x in rows], "win_rate:", round(sum(x>0 for x in rows)/len(rows), 2))
    return rows

before = preference_report(model, "before")

Loaded: HuggingFaceTB/SmolLM2-135M-Instruct
before margins: [-3.053, -2.853, -0.703] win_rate: 0.0


## 3. Compute fixed reference log-probs
Instead of keeping a second model in memory, this stores the initial reference scores before tuning.

In [13]:
model.eval()
ref_scores = []
with torch.no_grad():
    for p in pairs:
        pr = pref_prompt(p)
        ref_scores.append({
            "chosen": completion_logprob(model, tokenizer, pr, p["chosen"]).detach(),
            "rejected": completion_logprob(model, tokenizer, pr, p["rejected"]).detach(),
        })
print([{k: round(float(v.cpu()), 3) for k,v in row.items()} for row in ref_scores])

[{'chosen': -3.207, 'rejected': -10.789}, {'chosen': -3.344, 'rejected': -8.579}, {'chosen': -2.742, 'rejected': -8.929}]


## 4. Minimal DPO-style tuning loop

In [11]:
make_tiny_trainable(model)
model.train()
opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)
beta = 0.2
losses = []
for step in range(10):
    i = step % len(pairs)
    p = pairs[i]
    pr = pref_prompt(p)
    lp_c = completion_logprob(model, tokenizer, pr, p["chosen"])
    lp_r = completion_logprob(model, tokenizer, pr, p["rejected"])
    ref_c = ref_scores[i]["chosen"]
    ref_r = ref_scores[i]["rejected"]
    #margin = beta * ((lp_c - ref_c) - (lp_r - ref_r))
    margin = beta * (lp_c- lp_r)
    loss = -torch.nn.functional.logsigmoid(margin)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(float(loss.detach().cpu()))
print("DPO losses:", [round(x, 3) for x in losses])

Trainable tensors: 1 | trainable params: 28,311,552
DPO losses: [0.489, 0.583, 0.445, 0.358, 0.465, 0.364, 0.277, 0.373, 0.295, 0.217]


## 5. Preference report after tuning

In [12]:
after = preference_report(model, "after")
for p in pairs:
    print("\nPROMPT:", p["prompt"])
    print(generate_text(model, tokenizer, pref_prompt(p), max_new_tokens=40))

after margins: [7.582, 5.235, 6.187] win_rate: 1.0

PROMPT: Summarize this meeting for the team.
Instruction: Summarize this meeting for the team.
Answer: 1000 words

PROMPT: Route this support case.
Instruction: Route this support case.
Answer: 0

PROMPT: Summarize refund status for a support manager.
Instruction: Summarize refund status for a support manager.
Answer: 5


## Discussion
- Did the chosen-vs-rejected margin move in the expected direction?
- Which pair has the clearest label margin?
- What does this toy loop omit compared with production DPO?